<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/main/28_interpolation/10_lagrange_interpolation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


In [ ]:
# This cell is for the Google Colaboratory
# https://stackoverflow.com/a/63519730
if 'google.colab' in str(get_ipython()):
  path_py = '/content/nmisp_py'

  import os
  if not os.path.exists(path_py):
    import subprocess
    subprocess.run(
        ('git', 'clone', 'https://github.com/kwlee2025cpp/nmisp_py')
    )
  assert os.path.exists(path_py)

  import sys
  sys.path.insert(0, path_py)


In [ ]:
# 그래프, 수학 기능 추가
# Add graph and math features
import matplotlib.pyplot as plt
import numpy as np


# 라그랑주 내삽<br>Lagrange Interpolation


앞 노트북에서는 두 점 사이를 직선으로 잇는 **선형 내삽** 을 살펴보았다. 그렇다면 점이 세 개, 다섯 개, 그 이상이면 어떨까? 모든 점을 동시에 지나는 다항식이 있을까? 어떻게 만들 수 있을까?<br>
The previous notebook covered **linear interpolation** &mdash; a straight line between two points. What if we have three, five, or more data points? Is there a single polynomial that passes through *every* one of them &mdash; and how do we build it?

라그랑주 내삽은 이 질문에 대한 가장 깔끔한 답이다. 핵심은 *기저 다항식* 이라는 작은 도구이다.<br>
Lagrange interpolation is the cleanest answer to that question. The key idea is a small but powerful gadget called *basis polynomials*.


## 라그랑주 기저 다항식<br>Lagrange Basis Polynomials


$n+1$ 개의 자료점 $(x_0, y_0), (x_1, y_1), \ldots, (x_n, y_n)$ 이 있다고 하자. 각 자료점 $j$ 마다 다음과 같은 다항식을 정의한다.<br>
Suppose we have $n+1$ data points $(x_0, y_0), \ldots, (x_n, y_n)$. For each index $j$, define the polynomial

$$
L_j(x) \;=\; \prod_{\substack{i=0\\ i \neq j}}^{n} \frac{x - x_i}{x_j - x_i}.
$$

이 식의 가장 중요한 성질은 다음과 같다.<br>
The crucial property of $L_j(x)$ is

$$
L_j(x_i) \;=\; \delta_{ij} \;=\; \begin{cases} 1 & i = j \\ 0 & i \neq j \end{cases}
$$

자기 자신의 자료점 $x_j$ 에서는 1, 다른 자료점 $x_i$ ($i \ne j$) 에서는 0 이 된다. 분자에 $(x - x_i)$ 인수가 들어 있기 때문이다.<br>
At its own node $x_j$ it is 1; at every other node $x_i$ it is 0, because $(x - x_i)$ is one of the factors in the numerator.

이 단순한 성질이 *바로* 우리에게 필요한 것이다. 자세한 의미는 다음 절에서 보게 된다.<br>
That tiny identity is *exactly* what we need &mdash; the next section makes this concrete.


In [ ]:
def lagrange_basis(x_nodes, j, x):
    """j 번째 라그랑주 기저 다항식 L_j(x) 의 값을 계산.
       Evaluate the j-th Lagrange basis polynomial at x."""
    x_nodes = np.asarray(x_nodes, dtype=float)
    x = np.asarray(x, dtype=float)
    n = len(x_nodes)
    result = np.ones_like(x)
    for i in range(n):
        if i == j:
            continue
        result = result * (x - x_nodes[i]) / (x_nodes[j] - x_nodes[i])
    return result


기저 다항식이 어떻게 생겼는지 그려 보자.<br>
Let's actually look at what these basis polynomials look like.


In [ ]:
x_nodes = np.linspace(-1.0, 1.0, 5)
x_dense = np.linspace(-1.0, 1.0, 400)

plt.figure(figsize=(8, 4.5))
for j in range(len(x_nodes)):
    plt.plot(x_dense, lagrange_basis(x_nodes, j, x_dense), label=f'$L_{j}(x)$')

# 자료점 위치 표시 / mark node positions
for x_n in x_nodes:
    plt.axvline(x_n, color='gray', linestyle=':', alpha=0.5)
plt.axhline(1.0, color='gray', linestyle=':', alpha=0.3)
plt.axhline(0.0, color='gray', linestyle=':', alpha=0.3)

plt.title('5개의 절점에 대한 라그랑주 기저 / Lagrange basis on 5 equispaced nodes')
plt.xlabel('$x$')
plt.ylabel('$L_j(x)$')
plt.legend(loc='upper center', ncol=5)
plt.grid(True)
plt.show()


각 곡선이 *자기* 절점에서 정확히 1 이고 *다른* 절점에서 정확히 0 인 것을 눈으로 확인할 수 있다.<br>
Each curve passes through height 1 at *its own* node and height 0 at every *other* node &mdash; visible by eye.


## 보간 다항식 만들기<br>Constructing the Interpolant


기저 다항식을 갖춘 다음, $y$ 값으로 가중합을 만든다.<br>
Once we have the basis polynomials, we just take a weighted sum with the $y$ values:

$$
p_n(x) \;=\; \sum_{j=0}^{n} y_j \, L_j(x).
$$

왜 이것이 모든 자료점을 지나는가? 어떤 절점 $x_i$ 에 대입해 보자.<br>
Why does this pass through every data point? Plug in any node $x_i$:

$$
p_n(x_i) \;=\; \sum_{j=0}^{n} y_j \, L_j(x_i) \;=\; \sum_{j=0}^{n} y_j \, \delta_{ij} \;=\; y_i.
$$

합 안의 모든 항이 0 이고 단 하나 $j = i$ 인 항만 $y_i \cdot 1 = y_i$ 가 된다. *기저 자체가 보간 조건을 자동으로 만족시킨다.*<br>
Every term in the sum vanishes except the single $j = i$ term, which is $y_i \cdot 1 = y_i$. *The basis itself enforces the interpolation conditions, automatically.*


In [ ]:
def lagrange_interpolant(x_nodes, y_nodes, x):
    """라그랑주 보간 다항식 p_n(x) 값.
       Evaluate the Lagrange interpolant p_n(x)."""
    x_nodes = np.asarray(x_nodes, dtype=float)
    y_nodes = np.asarray(y_nodes, dtype=float)
    x = np.asarray(x, dtype=float)
    result = np.zeros_like(x)
    for j in range(len(x_nodes)):
        result = result + y_nodes[j] * lagrange_basis(x_nodes, j, x)
    return result


작은 예제로 확인해 보자: 4개의 자료점.<br>
A small sanity check on 4 data points.


In [ ]:
x_data = np.array([0.0, 1.0, 2.0, 4.0])
y_data = np.array([1.0, 2.0, 0.5, 3.0])

x_dense = np.linspace(x_data.min() - 0.5, x_data.max() + 0.5, 400)
y_dense = lagrange_interpolant(x_data, y_data, x_dense)

plt.figure(figsize=(8, 4.5))
plt.plot(x_dense, y_dense, label='Lagrange $p_3(x)$')
plt.plot(x_data, y_data, 'ko', markersize=8, label='data points')
plt.title('네 점을 지나는 3차 라그랑주 다항식 / Cubic Lagrange polynomial through 4 points')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.legend(); plt.grid(True)
plt.show()

# 자료점에서 정확히 일치하는지 확인 / verify exactness at the nodes
y_at_nodes = lagrange_interpolant(x_data, y_data, x_data)
print('p_n(x_data) =', y_at_nodes)
print('y_data      =', y_data)
print('max |error| =', np.max(np.abs(y_at_nodes - y_data)))


`numpy.polyfit(deg=n)` 으로 같은 다항식을 얻을 수도 있다 &mdash; 표현은 다르지만 결과 다항식은 동일해야 한다 (보간 다항식은 유일하다).<br>
We can recover the same polynomial via `numpy.polyfit(deg=n)` &mdash; different *representation*, identical *polynomial* (the interpolating polynomial through $n+1$ distinct nodes is unique).


In [ ]:
coeffs = np.polyfit(x_data, y_data, deg=3)
y_polyfit = np.polyval(coeffs, x_dense)
print('max |Lagrange - polyfit| =', np.max(np.abs(y_dense - y_polyfit)))


## 룽게 현상<br>The Runge Phenomenon


작은 차수에서는 이렇게 잘 작동하니, 차수를 더 올리면 더 좋아질 것 같다. 절점을 등간격으로 더 촘촘히 잡으면 어떻게 될까?<br>
Small-degree fits worked beautifully, so surely cranking up the degree (more equispaced nodes) only makes things better. Right?

룽게(Carl Runge, 1901) 가 다음 함수에서 직접 보였다.<br>
Carl Runge (1901) showed otherwise on the following function:

$$
f(x) \;=\; \frac{1}{1 + 25\,x^2}, \qquad x \in [-1, 1].
$$

매끄럽고 무해해 보이지만, 등간격 절점에서 차수가 커지면 양 끝에서 *심하게* 진동한다.<br>
It looks tame and smooth &mdash; yet equispaced high-degree interpolation oscillates *wildly* near the ends.


In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_dense = np.linspace(-1.0, 1.0, 800)
y_true = runge(x_dense)

plt.figure(figsize=(9, 5.5))
plt.plot(x_dense, y_true, 'k-', lw=2, label='$f(x)=1/(1+25x^2)$')

for n in (5, 10, 15, 20):
    x_nodes = np.linspace(-1.0, 1.0, n + 1)
    y_nodes = runge(x_nodes)
    y_interp = lagrange_interpolant(x_nodes, y_nodes, x_dense)
    plt.plot(x_dense, y_interp, label=f'$p_{{{n}}}$ (equispaced)')

plt.title('등간격 라그랑주 보간 / Equispaced Lagrange interpolation: Runge function')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.ylim(-2.0, 2.0)
plt.legend(); plt.grid(True)
plt.show()


$n = 20$ 에서는 양 끝의 진동이 함수 자체보다 *훨씬* 크다. 차수를 올릴수록 *더* 나빠진다.<br>
At $n = 20$ the end oscillations are *larger* than the function itself. Raising the degree makes things *worse*, not better.

**왜 이렇게 되는가?**<br>
**Why does this happen?**

* 등간격 절점에서의 보간 다항식의 최악의 보간 오차는 *르베그 상수* $\Lambda_n$ 의 영향을 받는다. 등간격 절점에서는 $\Lambda_n \sim 2^n / (e\, n \log n)$ 으로 *지수적으로* 증가한다. 따라서 절점 자료의 작은 진동도 양 끝에서 거대한 다항식 진동으로 증폭된다.<br>
  The worst-case interpolation error is governed by the *Lebesgue constant* $\Lambda_n$. For equispaced nodes, $\Lambda_n \sim 2^n / (e n \log n)$ &mdash; it grows *exponentially* in $n$. Tiny perturbations in node values get amplified into huge polynomial oscillations near the endpoints.

* 동등하게, 자료점에서 만들어지는 반더몬데 행렬 $V_{ij} = x_i^{\,j}$ 의 조건수가 $n$ 과 함께 폭발한다. 같은 정보로 점점 더 *민감한* 시스템을 풀게 된다.<br>
  Equivalently, the Vandermonde matrix $V_{ij} = x_i^{\,j}$ becomes increasingly *ill-conditioned* &mdash; we are solving a more and more sensitive linear system from the same information.

따라서 단순히 차수를 올리는 것은 답이 아니다. 두 가지 길이 있다: (1) 절점을 다르게 고르거나, (2) 다항식을 *구간별* 로 잘게 쪼갠다 (다음 노트북에서 다룰 스플라인).<br>
So &ldquo;just use more nodes&rdquo; is not the fix. The two real escapes are: (1) pick smarter nodes, or (2) chop the interval into pieces, each fit by a *low-degree* polynomial (splines &mdash; the next notebook).


### 동적 탐색<br>Interactive Exploration


$n$ 을 천천히 올려 가며 양 끝에서 진동이 커지는 것을 관찰해 보자.<br>
Drag $n$ slowly upward and watch the end oscillations swallow the function.


In [ ]:
import os
from ipywidgets import interact, IntSlider

_ci = bool(os.getenv("CI", False))


def plot_runge_lagrange(n):
    x_dense = np.linspace(-1.0, 1.0, 800)
    y_true = runge(x_dense)

    x_nodes = np.linspace(-1.0, 1.0, n + 1)
    y_nodes = runge(x_nodes)
    y_interp = lagrange_interpolant(x_nodes, y_nodes, x_dense)

    plt.figure(figsize=(8, 4.5))
    plt.plot(x_dense, y_true, 'k-', lw=2, label='$f$')
    plt.plot(x_dense, y_interp, 'r-', label=f'$p_{{{n}}}$ (equispaced)')
    plt.plot(x_nodes, y_nodes, 'ko')

    err_inf = np.max(np.abs(y_interp - y_true))
    plt.title(f'n = {n}, $\|p_n - f\|_\infty$ = {err_inf:.3g}')
    plt.xlabel('$x$'); plt.ylabel('$y$')
    plt.ylim(-2.0, 2.0)
    plt.legend(); plt.grid(True)
    plt.show()


if _ci:
    # 위젯 대신 첫 단계만 렌더 / Render only the first step instead of using widget
    plot_runge_lagrange(10)
else:
    interact(
        plot_runge_lagrange,
        n=IntSlider(min=2, max=30, step=1, value=10, description='n :'),
    );


## 체비셰프 절점<br>Chebyshev Nodes


절점을 등간격이 아니라 양 끝에 더 *조밀하게* 잡으면 어떻게 될까? 체비셰프 절점은 다음과 같이 정의된다.<br>
What if we cluster nodes more *densely* near the endpoints instead of spacing them evenly? The Chebyshev nodes on $[-1, 1]$ are

$$
x_k \;=\; \cos\!\left(\frac{2k+1}{2(n+1)}\pi\right), \qquad k = 0, 1, \ldots, n.
$$

이 절점에서는 르베그 상수가 $\Lambda_n \sim \frac{2}{\pi} \log n$ 로 *훨씬* 천천히 자란다. 결과는 다음과 같다.<br>
For these nodes the Lebesgue constant grows only as $\Lambda_n \sim \frac{2}{\pi} \log n$ &mdash; *logarithmically*, not exponentially. Watch what happens:


In [ ]:
def chebyshev_nodes(n, a=-1.0, b=1.0):
    """체비셰프 절점 (n+1 개) 을 [a, b] 위에 생성.
       Generate n+1 Chebyshev nodes on [a, b]."""
    k = np.arange(n + 1)
    t = np.cos((2*k + 1) * np.pi / (2*(n + 1)))   # nodes on [-1, 1]
    return 0.5*(a + b) + 0.5*(b - a) * t


x_dense = np.linspace(-1.0, 1.0, 800)
y_true = runge(x_dense)

plt.figure(figsize=(9, 5.5))
plt.plot(x_dense, y_true, 'k-', lw=2, label='$f$')

for n in (10, 20):
    x_nodes = chebyshev_nodes(n)
    y_nodes = runge(x_nodes)
    y_interp = lagrange_interpolant(x_nodes, y_nodes, x_dense)
    plt.plot(x_dense, y_interp, label=f'$p_{{{n}}}$ (Chebyshev)')

plt.title('체비셰프 절점에서의 라그랑주 보간 / Lagrange interpolation at Chebyshev nodes')
plt.xlabel('$x$'); plt.ylabel('$y$')
plt.ylim(-0.2, 1.2)
plt.legend(); plt.grid(True)
plt.show()


$n = 20$ 에서 등간격은 발산하지만, 체비셰프 절점은 함수와 거의 구분되지 않는다. *절점의 위치가 다항식 자체보다 더 중요할 수 있다.*<br>
At $n = 20$, equispaced nodes blow up; Chebyshev nodes are visually indistinguishable from $f$. *Node placement matters more than degree.*


## 언제 라그랑주를 쓰는가<br>When to Use Lagrange


| 상황<br>Situation | 추천<br>Recommendation |
|---|---|
| 자료점이 적음 ($n \lesssim 5$), 등간격<br>Few points ($n \lesssim 5$), equispaced | 라그랑주 OK |
| 자료점이 많음, 절점을 *고를 수 있음*<br>Many points, nodes are *yours to choose* | 라그랑주 + 체비셰프 절점 |
| 자료점이 많음, 절점이 *고정* (보통 측정값)<br>Many points, nodes are *fixed* (usually measurements) | 스플라인 (다음 노트북) / Splines (next notebook) |
| 미분이 매끄럽게 이어져야 함<br>Need a smooth derivative through the data | 3차 스플라인 / Cubic spline |

요약하면, 라그랑주는 *절점을 자유롭게 고를 수 있는* 함수 근사에는 강력하지만, 임의의 자료에 대한 일반적인 도구로는 한계가 있다. 다음 노트북에서 이 한계를 우회하는 *구간별 다항식* 인 스플라인을 다룬다.<br>
In short: Lagrange is powerful for *function approximation when nodes are yours to design*, but it is not the right hammer for arbitrary data. The next notebook covers the workhorse that escapes these limitations &mdash; *piecewise* polynomials, a.k.a. splines.


## 연습 문제<br>Exercises


Try this 1: $\sin\theta^\circ$ 를 $\theta = 0, 30, 60, \ldots, 180^\circ$ 에서 자료점으로 두고 라그랑주 다항식을 만들어 보시오. $\theta = 1^\circ$ 단위로 평가하여 정확한 $\sin$ 과 비교하시오.<br>
Build a Lagrange interpolant from $\sin\theta^\circ$ at $\theta = 0, 30, 60, \ldots, 180^\circ$. Evaluate it on a $1^\circ$ grid and compare with the true $\sin$ values.


Try this 2: 위 예제에서 $\theta$ 의 절점을 1, 11, 21, ..., $171^\circ$ 등으로 *비등간격* 으로 바꾸어 보시오. 결과가 어떻게 달라지는가?<br>
Repeat the previous exercise with *non-equispaced* nodes ($\theta = 1, 11, 21, \ldots, 171^\circ$). How does the result change?


Try this 3: $f(x) = e^x$ 를 $[-1, 1]$ 위에서 등간격 5개 절점과 체비셰프 5개 절점으로 보간해 보시오. 두 결과의 최대 오차 차이는 얼마인가?<br>
Interpolate $f(x) = e^x$ on $[-1, 1]$ using 5 equispaced nodes and 5 Chebyshev nodes. What is the difference in maximum error?


## 참고문헌<br>References


* R. L. Burden, J. D. Faires, A. M. Burden, *Numerical Analysis*, 10th Ed., Cengage, 2016 (Ch. 3 *Interpolation and Polynomial Approximation*).
* L. N. Trefethen, *Approximation Theory and Approximation Practice*, SIAM, 2013 (Ch. 5 *Barycentric interpolation formula*; Ch. 15 *Lebesgue constants*).
* C. Runge, &ldquo;&Uuml;ber empirische Funktionen und die Interpolation zwischen &auml;quidistanten Ordinaten,&rdquo; *Zeitschrift f&uuml;r Mathematik und Physik*, vol. 46, pp. 224&ndash;243, 1901.


## Final Bell<br>마지막 종


In [ ]:
# stackoverfow.com/a/24634221
import os
os.system("printf '\a'");
